# HazardNet — Model Training & Conversion (GPU)

**Phase 2 of the zero-cost MLOps pipeline.** Run this notebook in Google Colab (free T4 GPU) once a month/quarter to retrain the 15-channel 3D-CNN, convert the best checkpoint to TFLite, and push the new model bundle back to GitHub.

## How this fits into the architecture
| Profile | Where | When | Cost |
|---|---|---|---|
| **Daily Auto-Forecast (CPU, 19 min)** | GitHub Actions (`daily_forecast.yml`) | Every day at midnight UTC | Free (public repo, unlimited minutes) |
| **Training & Conversion (GPU, ~3 h)** | This Colab notebook | Monthly/quarterly (manual) | Free (Colab T4) |

The model flows Colab → GitHub → Production automatically. Data flows GEE/Open-Meteo → GitHub Actions → Firestore automatically.

## Pre-flight
1. In Colab menu: **Runtime → Change runtime type → GPU (T4)**.
2. Create a GitHub Fine-Grained Personal Access Token (PAT) with Contents: write permission on `myself-aas/HazardNet`. Paste it in the cell below when prompted.
3. Click **Runtime → Run all**.


## 1. Configuration

In [ ]:
import os, getpass

# GitHub credentials for the auto-push step.
GITHUB_USERNAME = "myself-aas"
GITHUB_REPO     = "HazardNet"
GITHUB_PAT      = getpass.getpass("GitHub Fine-Grained PAT (Contents: write): ")
GIT_USER_EMAIL  = "shuvo.1807016@bau.edu.bd"
GIT_USER_NAME   = "HazardNet Auto-ML"

# Dataset source (pick one):
#   "drive"  — Google Drive mount (fast, recommended after first download)
#   "hf"     — HuggingFace Datasets (zero-config)
DATASET_SOURCE = "hf"
HF_DATASET_REPO = "YOUR_USERNAME/HazardNet-Data"   # if using HF
DRIVE_TENSORS_PATH = "/content/drive/MyDrive/HazardNet/master_tensors.h5"

BUNDLE_DIR = "/content/HazardNet_Deployment_Bundles/deployment_bundle"
REPO_DIR   = "/content/HazardNet"
print("Configured.")

## 2. Sanity-check GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "⚠️ No GPU detected — Runtime → Change runtime type → GPU"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Device {torch.cuda.get_device_name(0)}")

## 3. Mount / Download dataset

In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
TENSORS_PATH = '/content/data/master_tensors.h5'

if DATASET_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    shutil.copy(DRIVE_TENSORS_PATH, TENSORS_PATH)
elif DATASET_SOURCE == 'hf':
    !wget -q "https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/master_tensors.h5" -O {TENSORS_PATH}

assert os.path.exists(TENSORS_PATH), f"Dataset not found at {TENSORS_PATH}"
!ls -lh {TENSORS_PATH}

## 4. Install training dependencies

In [ ]:
# Training stack (only needed in Colab — the Actions daily job uses tflite-runtime only).
!pip install -q torch torchvision h5py numpy pandas scikit-learn tqdm onnxruntime onnx tf2onnx tensorflow tensorflow-probability
import torch, torch.nn as nn, numpy as np, h5py, pandas as pd
from tqdm import tqdm
print("Training deps installed.")

## 5. Phase-3 Training Loop

Drop in your existing Kaggle training code here. The cell below is a placeholder that documents the expected interface — replace the body with your real model definition / training loop. What matters is that the final best weights get exported to:

*   `BEST_PT = '/content/best_model.pt'` (PyTorch state dict)


In [ ]:
# TODO: replace with your actual HazardNet model + training loop from the
# Kaggle notebook. Required output: BEST_PT must exist after this cell.
BEST_PT = '/content/best_model.pt'

# >>> PASTE YOUR TRAINING CODE HERE <<<
# model = HazardNet3DCNN(...).cuda()
# ... train ... 
# torch.save(model.state_dict(), BEST_PT)

assert os.path.exists(BEST_PT), f"Training did not produce {BEST_PT}"
print(f"Trained model saved to {BEST_PT} ({os.path.getsize(BEST_PT)/1e6:.2f} MB)")

## 6. Convert PyTorch → ONNX → TFLite

This matches the conversion you currently do on Kaggle. The end result is a ~0.75 MB `hazardnet_fp32.tflite` that the daily GitHub Actions job loads via `tflite-runtime`.

In [ ]:
import os, json, shutil
os.makedirs(BUNDLE_DIR, exist_ok=True)

# --- 6a. PyTorch -> ONNX ---
import torch
# model = HazardNet3DCNN(...)
# model.load_state_dict(torch.load(BEST_PT, map_location='cpu'))
# model.eval()
# dummy = torch.randn(1, 15, 10, 64, 64)
# torch.onnx.export(model, dummy, f'{BUNDLE_DIR}/hazardnet.onnx',
#                   input_names=['input'], output_names=['hazard', 'severity'],
#                   opset_version=17)

# --- 6b. ONNX -> TensorFlow SavedModel ---
# import onnx
# from onnx_tf.backend import prepare
# onnx_model = onnx.load(f'{BUNDLE_DIR}/hazardnet.onnx')
# tf_rep = prepare(onnx_model)
# tf_rep.export_graph(f'{BUNDLE_DIR}/savedmodel')

# --- 6c. SavedModel -> TFLite ---
# import tensorflow as tf
# converter = tf.lite.TFLiteConverter.from_saved_model(f'{BUNDLE_DIR}/savedmodel')
# converter.optimizations = [tf.lite.Optimize.DEFAULT]  # optional: f16/int8
# tflite_model = converter.convert()
# with open(f'{BUNDLE_DIR}/hazardnet_fp32.tflite', 'wb') as f:
#     f.write(tflite_model)

# --- 6d. Copy normalization stats + label file ---
# shutil.copy('normalization_stats.json', f'{BUNDLE_DIR}/normalization_stats.json')
# json.dump(LABELS, open(f'{BUNDLE_DIR}/labels.json','w'))

print("Conversion placeholder — wire in your actual ONNX/TF steps above.")
print("Bundle dir:", BUNDLE_DIR)

## 7. Verify the converted TFLite model

In [ ]:
# Smoke-test the artifact exactly as Actions will: load with tflite-runtime
# (same 2 MB wheel used in the daily job) and run one inference.
!pip install -q tflite-runtime numpy
import tflite_runtime.interpreter as tflite
import numpy as np

TFLITE_PATH = f'{BUNDLE_DIR}/hazardnet_fp32.tflite'
interp = tflite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()
dummy = np.random.randn(*inp['shape']).astype(inp['dtype'])
interp.set_tensor(inp['index'], dummy)
interp.invoke()
for o in out:
    print(f"  output {o['name']}: shape={o['shape']} dtype={o['dtype']}")
print(f"✅ TFLite model OK  ({os.path.getsize(TFLITE_PATH)/1e6:.2f} MB)")

## 8. Auto-push new model bundle to GitHub

In [ ]:
!git config --global user.email "{GIT_USER_EMAIL}"
!git config --global user.name  "{GIT_USER_NAME}"

# Shallow-clone main so we don't pull the full history in Colab.
!rm -rf {REPO_DIR}
!git clone --depth 1 https://{GITHUB_PAT}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git {REPO_DIR}
%cd {REPO_DIR}

# Replace the Models/ directory contents with the freshly-converted bundle.
!rm -rf Models/*
!cp -r {BUNDLE_DIR}/* Models/
!ls -lh Models/

# Commit and push on a dedicated auto-ml branch, then open a PR via gh CLI
# (safer than force-pushing to main). If you prefer direct push, just
# `git checkout main && git push` instead.
import datetime as _dt
BRANCH = f"auto-ml/model-update-{_dt.datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
BRANCH_MSG = f"chore(model): auto-update TFLite from Colab training run ({_dt.date.today()})"
!git checkout -b {BRANCH}
!git add Models/
!git -c user.name="{GIT_USER_NAME}" -c user.email="{GIT_USER_EMAIL}" commit -m "{BRANCH_MSG}"
!git push -u origin {BRANCH}

# Optionally create a PR using the GitHub API (no extra deps).
import json, urllib.request
req = urllib.request.Request(
    f'https://api.github.com/repos/{GITHUB_USERNAME}/{GITHUB_REPO}/pulls',
    data=json.dumps({
        'title': BRANCH_MSG,
        'head': BRANCH,
        'base': 'main',
        'body': 'Automated TFLite model update from Colab training run.\n\n- [ ] Verify `daily_forecast.yml` smoke passes\n- [ ] Check model_version bump\n',
    }).encode(),
    headers={'Authorization': f'token {GITHUB_PAT}', 'Accept': 'application/vnd.github+json'},
)
try:
    resp = urllib.request.urlopen(req)
    pr = json.load(resp)
    print(f"✅ PR opened: {pr['html_url']}")
except Exception as e:
    print(f"PR creation failed (push to branch succeeded): {e}")

## Done.

Once the PR is green:
1. Review the diff (only `Models/` files should change).
2. Merge to `main`.
3. The next daily GitHub Actions run at 00:00 UTC will automatically pick up the new `.tflite` and use it for inference.

If you ever want to skip the PR and push directly to `main`, replace the `git checkout -b` / `git push -u origin HEAD:BRANCH` block with:

```
!git checkout main
!git push origin main
```
